# 04 - 관리자의 Registry 승인 워크플로

이 Notebook에서는 두 가지 방법으로 Registry 레코드를 **승인하고 거부**하는 방법을 보여 줍니다.

1. **수동 승인** — AWS Agent Registry SDK를 사용해 대기 중인 레코드를 나열하고 검토한 후 사유와 함께 승인하거나 거부
2. **자동 승인** — Registry의 기본 제공 `autoApproval` 플래그를 사용하여 새 제출 항목을 자동으로 승인

## 학습 내용

- **관리자** 페르소나로 인증하고 기존 레지스트리 찾기
- 승인 대기 중인 레코드를 나열하고 세부 정보 검토
- SDK를 통해 상태 사유와 함께 레코드를 **승인** 또는 **거부**
- 자동화 데모를 위한 테스트 레코드 생성
- 새 제출 항목을 자동으로 승인하도록 레지스트리의 `autoApproval` 플래그 활성화
- autoApproval 워크플로의 **엔드 투 엔드 테스트** 실행
- 수동 승인과 autoApproval 방식 및 모범 사례 비교

## 사전 요구 사항

- boto3 >= 1.42.87
- 관리자, 게시자, 소비자 페르소나용 IAM 역할을 생성하려면 [Notebook 01](01-create-user-personas-workflow.ipynb)을 실행하세요.
- 관리자로 레지스트리를 생성하려면 [Notebook 02](02-creating-registry-workflow.ipynb)를 실행하세요.
- 게시자로 레지스트리에 레코드를 게시하려면 [Notebook 03](03-publishing-records-workflow.ipynb)을 실행하세요.

## 관리자 승인 워크플로
![관리자 승인 워크플로](images/admin_approval_flow_architecture.png)

## 관리자 승인 API 참조

| # | API | 설명 |
|---|-----|-------------|
| 1 | [ListRegistryRecords](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/list_registry_records.html) | 레지스트리의 모든 레코드를 나열하고 상태별로 필터링 |
| 2 | [GetRegistryRecord](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/get_registry_record.html) | 레코드 세부 정보를 검토하고 상태 변경 확인 |
| 3 | [UpdateRegistryRecordStatus](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/update_registry_record_status.html) | 상태 사유와 함께 대기 중인 레코드 승인 또는 거부 |
| 4 | [CreateRegistryRecord](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/create_registry_record.html) | 승인/거부 데모를 위한 테스트 레코드 생성 |
| 5 | [SubmitRegistryRecordForApproval](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/submit_registry_record_for_approval.html) | 관리자 검토를 위해 초안 레코드 제출 |
| 6 | [GetRegistry](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/get_registry.html) | 현재 레지스트리 구성(autoApproval 플래그) 확인 |
| 7 | [UpdateRegistry](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/update_registry.html) | 레지스트리의 autoApproval 설정 전환 |
| 8 | [DeleteRegistryRecord](https://docs.aws.amazon.com/boto3/latest/reference/services/bedrock-agentcore-control/client/delete_registry_record.html) | 데모 후 테스트 레코드 정리 |


### Notebook 진행 순서

02(레지스트리 생성) → 03(레코드 게시) → **04(이 Notebook)** → 05(시맨틱 검색)

#### 사용 사례: 엔터프라이즈 결제 처리
**관리자 페르소나:** AnyCompany의 레지스트리 관리자는 human-in-the-loop 승인 프로세스를 통해 제출된 레코드를 검토합니다. 각 제출 항목을 보안, 규정 준수, 품질 기준에 따라 평가한 후 승인 또는 거부 여부를 결정합니다. 승인된 기능은 검증을 거쳐 레지스트리에 게시되며 엔터프라이즈 전반의 권한 있는 AI 에이전트가 검색할 수 있습니다. 거부된 제출 항목은 수정할 수 있도록 피드백과 함께 개발 팀으로 반환됩니다.

---
## 1. boto3 SDK 및 종속성 설치

핵심 종속성(`boto3` 및 `python-dotenv`)을 설치합니다.

In [ ]:
!pip install boto3 python-dotenv --force-reinstall

## 2. 관리자로 boto3 Session 초기화

`admin_persona` IAM 역할을 수임하고 임시 자격 증명으로 boto3 Session을 생성합니다.

In [ ]:
import boto3
import json
import time
import os
import botocore.exceptions
import utils

AWS_REGION = os.environ.get("AWS_DEFAULT_REGION", "us-west-2")

# 현재 자격 증명에서 계정 ID 자동 감지
sts = boto3.client("sts", region_name=AWS_REGION)
ACCOUNT_ID = sts.get_caller_identity()["Account"]
CALLER_ARN = sts.get_caller_identity()["Arn"]

ADMIN_ROLE_ARN = f"arn:aws:iam::{ACCOUNT_ID}:role/admin_persona"

print(f"Account:  {ADMIN_ROLE_ARN}")

# 관리자 역할 수임
creds = utils.assume_role(
    role_arn=ADMIN_ROLE_ARN,
    session_name="admin-session",
)

admin_session = boto3.Session(
    aws_access_key_id=creds["AccessKeyId"],
    aws_secret_access_key=creds["SecretAccessKey"],
    aws_session_token=creds["SessionToken"],
    region_name=AWS_REGION,
)

## 3. Control Plane 클라이언트 초기화

Control plane(`bedrock-agentcore-control`)은 레지스트리와 레코드에 대한 CRUD 작업을 처리합니다.

In [ ]:
# Control plane 클라이언트(관리자 작업)
cp_client = admin_session.client("bedrock-agentcore-control")

---
## 4. 레지스트리 선택

사용할 기존 READY 레지스트리를 찾습니다. 위에서 `REGISTRY_ID`를 설정하면 해당 값을 검증합니다.
설정하지 않으면 `list_registries`에서 첫 번째 READY 레지스트리를 선택합니다.

READY 레지스트리를 찾을 수 없다면 먼저 **Notebook 02**를 실행하여 레지스트리를 생성해야 합니다.

In [ ]:
REGISTRY_ID = ""  # 위 목록에서 직접 선택하려면 이 값을 입력

## REGISTRY_ID가 비어 있으면 list_registries에서 첫 번째 READY 레지스트리를 선택
registry_details = utils.get_or_select_registry(cp_client, REGISTRY_ID, AWS_REGION)
REGISTRY_ID = registry_details[0]
REGISTRY_ARN = registry_details[1]

---
## 5. 기존 레코드 나열

Notebook 03에서 게시한 레코드를 포함하여 현재 레지스트리에 있는 모든 레코드를 나열합니다.
테스트 레코드를 생성하거나 승인 작업을 수행하기 전의 기준 상태를 확인할 수 있습니다.

In [ ]:
try:
    all_records = utils.list_records_with_ids(cp_client, REGISTRY_ID)

    print(f"Total records in registry: {len(all_records)}\n")
    for rec in all_records:
        status_marker = "🟡" if rec["status"] == "PENDING_APPROVAL" else "•"
        print(f"  {status_marker} [{rec['status']}] {rec['name']} ({rec['descriptorType']}) — {rec['recordId']}")

    pending = utils.filter_pending_records(all_records)
    print(f"\n📋 Records pending approval: {len(pending)}")
    for rec in pending:
        print(f"  • {rec['name']} | {rec['descriptorType']} | {rec['recordId']}")

    if not pending:
        print("\n   No records pending approval right now.")

except botocore.exceptions.ClientError as e:
    error_code = e.response["Error"]["Code"]
    if error_code == "ConflictException":
        print(f"❌ Registry {REGISTRY_ID} is not in READY state.")
        print("   Wait for the registry to finish provisioning or check its status.")
    else:
        print(f"❌ Error listing records: {error_code} — {e}")
    raise

Notebook 03을 실행했다면 위에서 `PENDING_APPROVAL` 상태의 레코드를 확인할 수 있습니다.

---
## 6. 수동 승인 워크플로

이 섹션에서는 수동 관리자 승인 프로세스를 보여 줍니다.
1. Notebook 03의 레코드 승인(Notebook 05에서 검색할 수 있도록 함)
2. 테스트 레코드를 생성하고 레코드 거부 흐름 확인

### 6a. Notebook 03의 레코드 승인

Notebook 03을 실행했다면 해당 레코드는 승인 대기 중입니다. Notebook 05에서 검색할 수 있도록 승인해 보겠습니다.
레코드는 `PENDING_APPROVAL` → `APPROVED`로 전환되며 소비자가 검색할 수 있게 됩니다.

In [ ]:
try:
    all_records = utils.list_records_with_ids(cp_client, REGISTRY_ID)
    pending = utils.filter_pending_records(all_records)

    # 테스트 레코드(admin_approval_test_*) 제외
    records_from_103 = [r for r in pending]

    if records_from_103:
        print(f"Found {len(records_from_103)} pending record(s) from notebook 03:\n")
        for rec in records_from_103:
            record_id = rec["recordId"]
            print(f"  Approving: {rec['name']} ({record_id})...")
            try:
                cp_client.update_registry_record_status(
                    registryId=REGISTRY_ID,
                    recordId=record_id,
                    status="APPROVED",
                    statusReason="Approved by admin in notebook 04 — record from notebook 03.",
                )
                verified = cp_client.get_registry_record(registryId=REGISTRY_ID, recordId=record_id)
                print(f"    ✅ Status: {verified['status']}")
            except botocore.exceptions.ClientError as e:
                error_code = e.response["Error"]["Code"]
                if error_code == "ConflictException":
                    print("    ⚠️  Record not in PENDING_APPROVAL state — may already be approved.")
                else:
                    print(f"    ❌ Error: {error_code} — {e}")
                    raise
    else:
        print("No records from notebook 03 found")

except botocore.exceptions.ClientError as e:
    error_code = e.response["Error"]["Code"]
    print(f"❌ Error listing records: {error_code} — {e}")
    raise

### 6b. 테스트 레코드 거부

먼저 거부 흐름을 보여 주기 위한 테스트 레코드를 생성한 다음 승인을 요청합니다.

In [ ]:
# 거부 테스트용 레코드 생성
try:
    reject_test_resp = cp_client.create_registry_record(
        registryId=REGISTRY_ID,
        name="admin_approval_test_reject",
        description="Test record created to demonstrate the admin rejection workflow.",
        descriptorType="CUSTOM",
        descriptors={
            "custom": {
                "inlineContent": json.dumps(
                    {
                        "type": "test-record",
                        "purpose": "rejection-demo",
                        "note": "This record intentionally has minimal metadata to demonstrate rejection.",
                    }
                )
            }
        },
        recordVersion="0.1",
    )
    REJECT_RECORD_ID = reject_test_resp["recordArn"].split("/")[-1]
    print(f"Created test record: {REJECT_RECORD_ID}")
    utils.wait_for_record_ready(cp_client, REGISTRY_ID, REJECT_RECORD_ID)

    # 거부할 수 있도록 승인 요청
    cp_client.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=REJECT_RECORD_ID)
    utils.wait_for_record_ready(cp_client, REGISTRY_ID, REJECT_RECORD_ID)
    print(f"Submitted for approval: {REJECT_RECORD_ID}")

except botocore.exceptions.ClientError as e:
    print(f"Error creating test record: {e}")
    raise

이제 테스트 레코드를 거부하여 거부 흐름을 확인합니다.
게시자는 거부 사유를 확인하고 그에 따라 레코드를 업데이트할 수 있습니다.

In [ ]:
if REJECT_RECORD_ID:
    try:
        reject_resp = cp_client.update_registry_record_status(
            registryId=REGISTRY_ID,
            recordId=REJECT_RECORD_ID,
            status="REJECTED",
            statusReason="Record rejected — missing detailed tool descriptions. Please update and resubmit.",
        )
        print(f"❌ Record {REJECT_RECORD_ID} rejected.")

        # 상태 변경 확인
        verified = cp_client.get_registry_record(registryId=REGISTRY_ID, recordId=REJECT_RECORD_ID)
        print(f"   Verified status: {verified['status']}")
        print(f"   Status reason:   {verified.get('statusReason', 'N/A')}")

    except botocore.exceptions.ClientError as e:
        error_code = e.response["Error"]["Code"]
        if error_code == "ConflictException":
            print(f"⚠️  Record {REJECT_RECORD_ID} is not in PENDING_APPROVAL state.")
            print("   It may have already been approved or rejected.")
        elif error_code == "ValidationException":
            print(f"⚠️  Record {REJECT_RECORD_ID} is not in PENDING_APPROVAL state.")
            print("   Make sure you ran Section 6 (submit for approval) first.")
        else:
            print(f"❌ Error rejecting record: {error_code} — {e}")
            raise
else:
    print("No reject test record found. Run Section 6 first.")

### 모든 레코드 나열

위의 승인 및 거부 작업을 수행한 후 레지스트리의 모든 레코드를 확인합니다.

In [ ]:
all_records = utils.list_records_with_ids(cp_client, REGISTRY_ID)

print(f"Total records in registry: {len(all_records)}\n")
for rec in all_records:
    status_marker = "🟡" if rec["status"] == "PENDING_APPROVAL" else "•"
    print(f"  {status_marker} [{rec['status']}] {rec['name']} ({rec['descriptorType']}) — {rec['recordId']}")

---
## 7. 자동 승인(autoApproval 플래그)

AWS Agent Registry는 기본 제공 **autoApproval** 플래그를 제공합니다. 이 플래그를 활성화하면 승인을 요청한 모든 새 레코드가
수동 개입 없이 자동으로 승인됩니다.

**권장 사항:** 프로덕션 워크로드에서는 레코드를 Registry에서 검색할 수 있게 만들기 전에 철저히 검토하는 것이 좋습니다.

워크플로:
1. 현재 레지스트리 구성 표시(`autoApproval`은 Notebook 02의 설정에 따라 `False`여야 함)
2. 레지스트리를 업데이트하여 `autoApproval: True` 설정
3. 새 테스트 레코드를 생성하고 승인 요청
4. 자동으로 승인되었는지 확인하기 위해 레지스트리 레코드 나열
5. 레지스트리를 다시 `autoApproval: False`로 복원

### 7a. 현재 레지스트리 구성 표시

변경하기 전에 현재 `autoApproval` 설정을 확인합니다.

In [ ]:
try:
    registry_config = cp_client.get_registry(registryId=REGISTRY_ID)
    current_auto_approval = registry_config.get("approvalConfiguration", {}).get("autoApproval", False)

    print(f"Registry: {registry_config.get('name', 'N/A')} ({REGISTRY_ID})")
    print(f"Status:   {registry_config.get('status', 'N/A')}")
    print("\nCurrent approvalConfiguration:")
    print(f"  autoApproval: {current_auto_approval}")

    if current_auto_approval:
        print("\n⚠️  autoApproval is already True. Records are being auto-approved.")
        print("   We will still demonstrate the workflow below.")
    else:
        print("\n✅ autoApproval is False — manual approval is required.")
        print("   We will enable it in the next step.")

except botocore.exceptions.ClientError as e:
    error_code = e.response["Error"]["Code"]
    print(f"❌ Error getting registry config: {error_code} — {e}")
    raise

### 7b. autoApproval 활성화

레지스트리를 업데이트하여 `autoApproval: True`로 설정합니다. 이 변경 후 승인을 요청한 모든 새 레코드는
서비스에서 자동으로 승인됩니다.

In [ ]:
try:
    update_resp = cp_client.update_registry(
        registryId=REGISTRY_ID,
        approvalConfiguration={"optionalValue": {"autoApproval": True}},
    )

    print("✅ Registry updated — autoApproval is now enabled.")
    print(f"   Updated at: {update_resp.get('updatedAt', 'N/A')}")

    # 변경 사항 확인
    verify_config = cp_client.get_registry(registryId=REGISTRY_ID)
    verified_auto = verify_config.get("approvalConfiguration", {}).get("autoApproval", False)
    print(f"   Verified autoApproval: {verified_auto}")

    while True:
        r = cp_client.get_registry(registryId=REGISTRY_ID)
        if r["status"] == "READY":
            print("Registry is READY")
            break
        print(f"Status: {r['status']} - waiting...")
        time.sleep(3)

except botocore.exceptions.ClientError as e:
    error_code = e.response["Error"]["Code"]
    if error_code == "ConflictException":
        print(f"❌ Registry {REGISTRY_ID} is not in READY state.")
        print("   The registry must be in READY state before it can be updated.")
        print("   Wait for the registry to finish provisioning and re-run this cell.")
    elif error_code == "AccessDeniedException":
        print(f"❌ Access denied: {e}")
        print("   Verify admin_persona has bedrock-agentcore:UpdateRegistry permission.")
        raise
    else:
        print(f"❌ Error updating registry: {error_code} — {e}")
        raise

### 7c. 테스트 레코드 생성 및 제출(autoApproval 활성화)

새 테스트 A2A 레코드를 생성하고 승인을 요청합니다. `autoApproval: True`이므로
레코드는 수동 개입 없이 자동으로 승인되어야 합니다.

### 자동 승인이 활성화된 레지스트리에 레코드 추가

In [ ]:
from botocore.exceptions import ClientError

a2a_agent_card = json.dumps(
    {
        "protocolVersion": "0.3.0",
        "name": "Refund Processing Agent",
        "description": "Handles payment refunds",
        "url": "https://example.com/agents/refund",
        "version": "1.0.0",
        "capabilities": {"streaming": True},
        "defaultInputModes": ["text"],
        "defaultOutputModes": ["text"],
        "preferredTransport": "JSONRPC",
        "skills": [
            {
                "id": "refund_handling",
                "name": "Refund Handling",
                "description": "Handle refunds",
                "tags": [],
            }
        ],
    }
)


try:
    a2a_resp = cp_client.create_registry_record(
        registryId=REGISTRY_ID,
        name="a2a_refund_agent_auto_approval",
        description="A2A agent for refunds",
        descriptorType="A2A",
        descriptors={
            "a2a": {
                "agentCard": {
                    "schemaVersion": "0.3",
                    "inlineContent": a2a_agent_card,
                }
            }
        },
        recordVersion="1.0",
    )
    A2A_RECORD_ID = a2a_resp["recordArn"].split("/")[-1]  # registryRecordArn이 아니라 recordArn
    print(f"Created A2A record: {A2A_RECORD_ID}")
    utils.wait_for_record_ready(cp_client, REGISTRY_ID, A2A_RECORD_ID)

except ClientError as e:
    if e.response["Error"]["Code"] == "ConflictException":
        records = cp_client.list_registry_records(registryId=REGISTRY_ID)
        for rec in records.get("registryRecords", []):
            if rec["name"] == "payment_agent":
                A2A_RECORD_ID = rec["recordId"]  # registryRecordId가 아니라 recordId
                break
        print(f"  Using existing record: {A2A_RECORD_ID}")
    else:
        raise

print(f"\nA2A_RECORD_ID = {A2A_RECORD_ID}")

### 자동 승인을 위한 레코드 제출

In [ ]:
# 레코드가 준비될 때까지 기다린 후 승인 요청
try:
    print(f"Waiting for record {A2A_RECORD_ID} to be ready...")
    ready_resp = utils.wait_for_record_ready(cp_client, REGISTRY_ID, A2A_RECORD_ID)

    if ready_resp["status"] == "DRAFT":
        cp_client.submit_registry_record_for_approval(registryId=REGISTRY_ID, recordId=A2A_RECORD_ID)
        print("✅ Record submitted for approval — autoApproval should handle it automatically.")
    elif ready_resp["status"] == "PENDING_APPROVAL":
        print("Record is already PENDING_APPROVAL.")
    elif ready_resp["status"] == "APPROVED":
        print("Record is already APPROVED — autoApproval may have already processed it.")
    else:
        print(f"Record is in {ready_resp['status']} state. Run cleanup first.")

except TimeoutError as e:
    print(f"⏰ {e}")
except botocore.exceptions.ClientError as e:
    error_code = e.response["Error"]["Code"]
    if error_code == "ConflictException":
        print("⚠️  Record not in DRAFT state — may already be submitted.")
        current = cp_client.get_registry_record(registryId=REGISTRY_ID, recordId=A2A_RECORD_ID)
        print(f"   Current status: {current['status']}")
    else:
        raise

### 7d. 자동 승인 확인

테스트 레코드가 자동으로 승인되었는지 확인하기 위해 레지스트리 레코드를 나열합니다.
`autoApproval: True`이면 제출 후 레코드가 수동 개입 없이
곧바로 `APPROVED`로 전환되어야 합니다.

In [ ]:
# 자동 승인 처리가 완료될 때까지 폴링(최대 60초)
print("Polling for auto-approval (every 5s, up to 60s)...")
poll_interval = 5
poll_timeout = 60
elapsed = 0
final_status = None

try:
    while elapsed < poll_timeout:
        auto_record = cp_client.get_registry_record(registryId=REGISTRY_ID, recordId=A2A_RECORD_ID)
        final_status = auto_record.get("status")
        print(f"  [{elapsed}s] Status: {final_status}")

        if final_status != "PENDING_APPROVAL":
            break

        time.sleep(poll_interval)
        elapsed += poll_interval

    # print(f"\nRecord: {AUTO_APPROVAL_RECORD_NAME}")
    print(f"  Record ID:     {A2A_RECORD_ID}")
    print(f"  Final status:  {final_status}")
    print(f"  Status reason: {auto_record.get('statusReason', 'N/A')}")

    if final_status == "APPROVED":
        print("\n\u2705 Auto-approval worked! The record was approved automatically.")
        print("   No manual intervention was needed.")
    elif final_status == "PENDING_APPROVAL":
        print(f"\n\u23f3 Record is still PENDING_APPROVAL after {poll_timeout}s.")
        print("   Try re-running this cell, or check that autoApproval was enabled")
        print("   correctly in Section 7b.")
    else:
        print(f"\n\u26a0\ufe0f  Unexpected status: {final_status}")

except botocore.exceptions.ClientError as e:
    error_code = e.response["Error"]["Code"]
    print(f"\u274c Error checking record status: {error_code} \u2014 {e}")
    raise

# 전체 상태를 확인할 수 있도록 모든 레코드도 나열
print("\n--- All Registry Records ---")
all_records_after = utils.list_records_with_ids(cp_client, REGISTRY_ID)
for rec in all_records_after:
    print(f"  [{rec['status']}] {rec['name']} \u2014 {rec['recordId']}")

### 7e. 자동 승인 테스트 레코드 삭제

승인 구성을 복원하기 전에 레지스트리를 깔끔하게 유지할 수 있도록 7c에서 생성한 테스트 레코드를 정리합니다.

In [ ]:
# 자동 승인 테스트 레코드 삭제
try:
    cp_client.delete_registry_record(registryId=REGISTRY_ID, recordId=A2A_RECORD_ID)
    print(f"✅ Deleted auto-approval test record: {A2A_RECORD_ID}")
except botocore.exceptions.ClientError as e:
    error_code = e.response["Error"]["Code"]
    if error_code == "ResourceNotFoundException":
        print(f"Record {A2A_RECORD_ID} already deleted.")
    else:
        print(f"❌ Error deleting record: {error_code} — {e}")
        raise

### 7f. autoApproval을 False로 복원

다른 Notebook과 프로덕션 워크플로가 예상대로 계속 작동하도록 레지스트리를 수동 승인이 필요한 상태로 복원합니다.

In [ ]:
try:
    restore_resp = cp_client.update_registry(
        registryId=REGISTRY_ID,
        approvalConfiguration={"optionalValue": {"autoApproval": False}},
    )

    print("✅ Registry restored — autoApproval is now disabled.")
    print(f"   Updated at: {restore_resp.get('updatedAt', 'N/A')}")

    # 변경 사항 확인
    verify_restore = cp_client.get_registry(registryId=REGISTRY_ID)
    verified_auto = verify_restore.get("approvalConfiguration", {}).get("autoApproval", False)
    print(f"   Verified autoApproval: {verified_auto}")

except botocore.exceptions.ClientError as e:
    error_code = e.response["Error"]["Code"]
    print(f"❌ Error restoring registry: {error_code} — {e}")
    print("   ⚠️  IMPORTANT: Manually set autoApproval back to False to avoid")
    print("   unintended auto-approvals in other notebooks.")
    print("   The cleanup section (Section 8) will also attempt to restore this setting.")
    raise

## 사전 요구 Notebook
- **Notebook 01** — [사용자 페르소나 생성](01-create-user-personas-workflow.ipynb): 관리자, 게시자, 소비자 사용자 페르소나 설정
- **Notebook 02** — [레지스트리 생성](02-creating-registry-workflow.ipynb): 관리자가 레지스트리 생성
- **Notebook 03** — [레코드 게시](03-publishing-records-workflow.ipynb): 게시자로 레코드 게시

## 다음 단계
- **Notebook 05** — [시맨틱 검색](05-search-registry-workflow.ipynb): 소비자로서 NLQ를 사용해 승인된 레코드 검색